In [2]:
#Poblar tablas de SQL fleetlogix 

"""
FleetLogix - Generación de Datos Sintéticos Completos
Genera 505000+ registros respetando relaciones y reglas de negocio
"""

import psycopg2
from psycopg2.extras import execute_batch
import pandas as pd
import numpy as np
from faker import Faker
from datetime import datetime, timedelta
import random
import logging
from tqdm import tqdm
import json

# Configuración de logging
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(levelname)s - %(message)s',
    handlers=[
        logging.FileHandler('data_generation.log'),
        logging.StreamHandler()
    ]
)

# Configuración de conexión
DB_CONFIG = {
    'host': 'localhost',
    'database': 'Proyectinegrador',
    'user': 'postgres',
    'password': 'mementomori',  # <-- ¡Recuerda cambiar esto!
    'port': 5432
}

# Inicializar Faker con semilla para reproducibilidad
fake = Faker('es_CO')  # Español Colombia (por Bogotá)
Faker.seed(42)
random.seed(42)
np.random.seed(42)

class DataGenerator:
    def __init__(self, db_config):
        self.db_config = db_config
        self.connection = None
        self.cursor = None
        self.cities = ['Bogotá', 'Medellín', 'Cali', 'Barranquilla', 'Cartagena']
        
        # Contadores para logging
        self.counters = {
            'vehicles': 0,
            'drivers': 0,
            'routes': 0,
            'trips': 0,
            'deliveries': 0,
            'maintenance': 0
        }
    
    def connect(self):
        """Establecer conexión con la base de datos"""
        try:
            self.connection = psycopg2.connect(**self.db_config)
            self.cursor = self.connection.cursor()
            logging.info(" Conexión exitosa a PostgreSQL")
            return True
        except Exception as e:
            logging.error(f" Error al conectar: {e}")
            return False
    
    def generate_vehicles(self, count=200):
        """Generar 200 vehículos con diferentes tipos y capacidades"""
        logging.info(f"Generando {count} vehículos...")
        
        vehicle_types = [
            ('Camión Grande', 5000, 'diesel', 0.3),
            ('Camión Mediano', 3000, 'diesel', 0.3),
            ('Van', 1500, 'gasolina', 0.3),
            ('Motocicleta', 50, 'gasolina', 0.1)
        ]
        
        vehicles = []
        for i in range(count):
            v_type, capacity, fuel, prob = random.choices(
                vehicle_types, 
                weights=[vt[3] for vt in vehicle_types]
            )[0]
            
            # Generar placa colombiana (ABC123)
            license_plate = f"{fake.random_uppercase_letter()}{fake.random_uppercase_letter()}{fake.random_uppercase_letter()}{random.randint(100,999)}"
            
            # Fecha de adquisición en los últimos 5 años
            acquisition_date = fake.date_between(start_date='-5y', end_date='-1m')
            
            # 90% activos, 10% en mantenimiento o inactivos
            status = random.choice(['active'] * 9 + ['maintenance'])
            
            vehicles.append((
                license_plate,
                v_type,
                capacity,
                fuel,
                acquisition_date,
                status
            ))
        
        # Insertar en batch (SQL Corregido)
        query = """
            INSERT INTO vehicles (license_plate, vehicle_type, capacity_kg, 
                                fuel_type, acquisition_date, status)
            VALUES (%s, %s, %s, %s, %s, %s)
        """
        
        execute_batch(self.cursor, query, vehicles, page_size=100)
        self.connection.commit()
        self.counters['vehicles'] = count
        logging.info(f" {count} vehículos insertados")
    
    def generate_drivers(self, count=400):
        """Generar 400 conductores con datos realistas"""
        logging.info(f"Generando {count} conductores...")
        
        drivers = []
        license_types = ['C1', 'C2', 'C3', 'A2']
        
        for i in range(count):
            employee_code = f"EMP{str(i+1).zfill(4)}"
            first_name = fake.first_name()
            last_name = fake.last_name()
            
            # Licencia colombiana
            license_number = f"{random.randint(1000000000, 9999999999)}"
            license_type = random.choice(license_types)
            
            # Licencia válida por 3 años, algunas próximas a vencer
            license_expiry = fake.date_between(start_date='-1m', end_date='+3y')
            
            phone = f"3{random.randint(100000000, 999999999)}"
            
            # Fecha de contratación en los últimos 5 años
            hire_date = fake.date_between(start_date='-5y', end_date='-1w')
            
            # 95% activos
            status = random.choice(['active'] * 19 + ['inactive'])
            
            drivers.append((
                employee_code,
                first_name,
                last_name,
                license_number,
                license_expiry,
                phone,
                hire_date,
                status
            ))
        
        query = """
            INSERT INTO drivers (employee_code, first_name, last_name, 
                               license_number, license_expiry, phone, 
                               hire_date, status)
            VALUES (%s, %s, %s, %s, %s, %s, %s, %s)
        """
        
        execute_batch(self.cursor, query, drivers, page_size=100)
        self.connection.commit()
        self.counters['drivers'] = count
        logging.info(f" {count} conductores insertados")
    
    def generate_routes(self, count=50):
        """Generar 50 rutas entre las 5 ciudades principales"""
        logging.info(f"Generando {count} rutas...")
        
        routes = []
        route_counter = 1
        
        for origin in self.cities:
            for destination in self.cities:
                if origin != destination:
                    num_routes = 3 if origin == 'Bogotá' or destination == 'Bogotá' else 2
                    
                    for i in range(num_routes):
                        route_code = f"R{str(route_counter).zfill(3)}"
                        base_distance = self._get_distance(origin, destination)
                        distance = base_distance + random.uniform(-50, 50)
                        
                        avg_speed = random.uniform(60, 80)
                        duration = distance / avg_speed
                        
                        toll_cost = int(distance / 100) * 15000
                        
                        routes.append((
                            route_code,
                            origin,
                            destination,
                            round(distance, 2),
                            round(duration, 2),
                            toll_cost
                        ))
                        
                        route_counter += 1
                        if route_counter > count:
                            break
                    
                    if route_counter > count:
                        break
            
            if route_counter > count:
                break
        
        routes = routes[:count]
        
        query = """
            INSERT INTO routes (route_code, origin_city, destination_city, 
                              distance_km, estimated_duration_hours, toll_cost)
            VALUES (%s, %s, %s, %s, %s, %s)
        """
        
        execute_batch(self.cursor, query, routes, page_size=50)
        self.connection.commit()
        self.counters['routes'] = count
        logging.info(f" {count} rutas insertadas")
    
    def _get_distance(self, origin, destination):
        """Obtener distancia aproximada entre ciudades colombianas"""
        distances = {
            ('Bogotá', 'Medellín'): 440,
            ('Bogotá', 'Cali'): 460,
            ('Bogotá', 'Barranquilla'): 1000,
            ('Bogotá', 'Cartagena'): 1050,
            ('Medellín', 'Cali'): 420,
            ('Medellín', 'Barranquilla'): 640,
            ('Medellín', 'Cartagena'): 640,
            ('Cali', 'Barranquilla'): 1100,
            ('Cali', 'Cartagena'): 1100,
            ('Barranquilla', 'Cartagena'): 120
        }
        
        key = tuple(sorted([origin, destination]))
        return distances.get(key, 500)
    
    def generate_trips(self, count=100000):
        """Generar 100000 viajes en 2 años de operación"""
        logging.info(f"Generando {count} viajes...")
        
        self.cursor.execute("SELECT vehicle_id, capacity_kg FROM vehicles WHERE status = 'active'")
        vehicles = self.cursor.fetchall()
        
        self.cursor.execute("SELECT driver_id FROM drivers WHERE status = 'active'")
        drivers = [d[0] for d in self.cursor.fetchall()]
        
        self.cursor.execute("SELECT route_id, distance_km, estimated_duration_hours FROM routes")
        routes = self.cursor.fetchall()
        
        start_date = datetime.now() - timedelta(days=730)
        trips = []
        current_date = start_date
        
        for i in tqdm(range(count), desc="Generando trips"):
            vehicle_id, capacity = random.choice(vehicles)
            capacity = float(capacity)
            driver_id = random.choice(drivers)
            route_id, distance, est_duration = random.choice(routes)
            
            distance = float(distance)
            est_duration = float(est_duration)
            
            hour = np.random.choice(range(24), p=self._get_hourly_distribution())
            departure = current_date.replace(hour=hour, minute=random.randint(0, 59))
            
            actual_duration = est_duration * random.uniform(0.8, 1.3)
            arrival = departure + timedelta(hours=actual_duration)
            
            fuel_consumed = distance * random.uniform(0.08, 0.15)
            total_weight = capacity * random.uniform(0.4, 0.9)
            
            if arrival < datetime.now():
                status = 'completed'
            else:
                status = 'in_progress'
            
            trips.append((
                vehicle_id,
                driver_id,
                route_id,
                departure,
                arrival if status == 'completed' else None,
                round(fuel_consumed, 2),
                round(total_weight, 2),
                status
            ))
            
            current_date += timedelta(minutes=int(1440 * 2 * 365 / count))
        
        query = """
            INSERT INTO trips (vehicle_id, driver_id, route_id, departure_datetime,
                             arrival_datetime, fuel_consumed_liters, total_weight_kg, status)
            VALUES (%s, %s, %s, %s, %s, %s, %s, %s)
        """
        
        batch_size = 1000
        for i in range(0, len(trips), batch_size):
            batch = trips[i:i+batch_size]
            execute_batch(self.cursor, query, batch, page_size=100)
            self.connection.commit()
            
            if i % 10000 == 0:
                logging.info(f"  Progreso: {i}/{count} trips insertados")
        
        self.counters['trips'] = count
        logging.info(f" {count} viajes insertados")
    
    def _get_hourly_distribution(self):
        """Distribución de probabilidad por hora del día"""
        probs = np.ones(24) * 0.02
        probs[6:20] = 0.06
        probs[8:12] = 0.08
        probs[14:18] = 0.07
        return probs / probs.sum()
    
    def generate_deliveries(self, count=400000):
        """Generar 400000 entregas (promedio 4 por viaje)"""
        logging.info(f"Generando {count} entregas...")
        
        # Obtener todos los trips con su información (SQL Corregido)
        self.cursor.execute("""
            SELECT 
                t.trip_id, 
                t.departure_datetime, 
                t.arrival_datetime, 
                t.total_weight_kg,
                r.destination_city
            FROM trips t
            JOIN routes r ON t.route_id = r.route_id
        """)
        trips_data = self.cursor.fetchall()
        
        deliveries = []
        delivery_counter = 0
        
        for trip_id, departure, arrival, total_weight, city in tqdm(trips_data, desc="Generando deliveries"):
            num_deliveries = np.random.choice([2, 3, 4, 5, 6], p=[0.1, 0.2, 0.4, 0.2, 0.1])
            
            # Peso por entrega (Llamada corregida a la función matemática)
            weights = self._distribute_weight(float(total_weight), num_deliveries)
            
            if arrival:
                delivery_duration = (arrival - departure).total_seconds() / 3600
                time_per_delivery = delivery_duration / num_deliveries
            else:
                time_per_delivery = 0.5
            
            for i in range(num_deliveries):
                tracking_number = f"FL{datetime.now().year}{str(delivery_counter+1).zfill(8)}"
                customer_name = fake.name()
                delivery_address = f"{fake.street_address()}, {city}"
                package_weight = weights[i]
                
                scheduled = departure + timedelta(hours=time_per_delivery * (i + 0.5))
                
                if arrival:
                    if random.random() < 0.9:
                        delivered = scheduled + timedelta(minutes=random.randint(-30, 30))
                    else:
                        delivered = scheduled + timedelta(minutes=random.randint(60, 180))
                    
                    delivery_status = 'delivered'
                    signature = random.random() < 0.95
                else:
                    delivered = None
                    delivery_status = 'pending'
                    signature = False
                
                deliveries.append((
                    trip_id,
                    tracking_number,
                    customer_name,
                    delivery_address,
                    round(package_weight, 2),
                    scheduled,
                    delivered,
                    delivery_status,
                    signature
                ))
                
                delivery_counter += 1
                if delivery_counter >= count:
                    break
            
            if delivery_counter >= count:
                break
        
        query = """
            INSERT INTO deliveries (trip_id, tracking_number, customer_name,
                                  delivery_address, package_weight_kg, scheduled_datetime,
                                  delivered_datetime, delivery_status, recipient_signature)
            VALUES (%s, %s, %s, %s, %s, %s, %s, %s, %s)
        """
        
        batch_size = 1000
        for i in range(0, len(deliveries), batch_size):
            batch = deliveries[i:i+batch_size]
            execute_batch(self.cursor, query, batch, page_size=100)
            self.connection.commit()
            
            if i % 50000 == 0:
                logging.info(f"  Progreso: {i}/{len(deliveries)} deliveries insertados")
        
        self.counters['deliveries'] = len(deliveries)
        logging.info(f" {len(deliveries)} entregas insertadas")
    
    def _distribute_weight(self, total_weight, num_packages):
        """Distribuir peso total entre paquetes de manera realista (Función Corregida)"""
        weights_proportions = np.random.dirichlet(np.ones(num_packages))
        weights = weights_proportions * (total_weight * 0.95)
        weights = np.maximum(weights, 0.5)
        return weights.tolist()
    
    def generate_maintenance(self, count=5000):
        """Generar 5000 registros de mantenimiento"""
        logging.info(f"Generando {count} registros de mantenimiento...")
        
        # Obtener información de vehículos y sus viajes (SQL Corregido)
        self.cursor.execute("""
            SELECT 
                v.vehicle_id,
                v.vehicle_type,
                COUNT(t.trip_id) as trip_count,
                MIN(t.departure_datetime) as first_trip,
                MAX(t.arrival_datetime) as last_trip
            FROM vehicles v
            LEFT JOIN trips t ON v.vehicle_id = t.vehicle_id
            GROUP BY v.vehicle_id, v.vehicle_type
        """)
        vehicle_stats = self.cursor.fetchall()
        
        maintenance_types = [
            ('Cambio de aceite', 150000, 30),
            ('Revisión de frenos', 250000, 60),
            ('Cambio de llantas', 450000, 90),
            ('Mantenimiento general', 350000, 45),
            ('Revisión de motor', 500000, 60),
            ('Alineación y balanceo', 180000, 30)
        ]
        
        maintenance_records = []
        
        for vehicle_id, vehicle_type, trip_count, first_trip, last_trip in vehicle_stats:
            num_maintenance = max(1, trip_count // 20)
            
            if first_trip and last_trip:
                operation_days = (last_trip - first_trip).days
                
                for i in range(min(num_maintenance, count - len(maintenance_records))):
                    days_offset = int(operation_days * (i + 1) / (num_maintenance + 1))
                    maintenance_date = (first_trip + timedelta(days=days_offset)).date()
                    
                    maint_type, base_cost, days_next = random.choice(maintenance_types)
                    cost = base_cost * random.uniform(0.8, 1.2)
                    description = f"{maint_type} programado para {maintenance_date.strftime('%Y-%m-%d')}"
                    next_maintenance = maintenance_date + timedelta(days=days_next)
                    performed_by = f"{fake.first_name()} {fake.last_name()}"
                    
                    maintenance_records.append((
                        vehicle_id,
                        maintenance_date,
                        maint_type,
                        description,
                        round(cost, 2),
                        next_maintenance,
                        performed_by
                    ))
                    
                    if len(maintenance_records) >= count:
                        break
        
        query = """
            INSERT INTO maintenance (vehicle_id, maintenance_date, maintenance_type,
                                   description, cost, next_maintenance_date, performed_by)
            VALUES (%s, %s, %s, %s, %s, %s, %s)
        """
        
        execute_batch(self.cursor, query, maintenance_records[:count], page_size=100)
        self.connection.commit()
        self.counters['maintenance'] = min(len(maintenance_records), count)
        logging.info(f" {self.counters['maintenance']} mantenimientos insertados")
    
    def validate_data_quality(self):
        """Validar integridad y calidad de datos"""
        logging.info("\n VALIDANDO CALIDAD DE DATOS...")
        
        validations = {
            "Integridad referencial - Trips sin vehículo válido": """
                SELECT COUNT(*) FROM trips t 
                LEFT JOIN vehicles v ON t.vehicle_id = v.vehicle_id 
                WHERE v.vehicle_id IS NULL
            """,
            "Integridad referencial - Deliveries sin trip válido": """
                SELECT COUNT(*) FROM deliveries d 
                LEFT JOIN trips t ON d.trip_id = t.trip_id 
                WHERE t.trip_id IS NULL
            """,
            "Consistencia temporal - Trips con arrival < departure": """
                SELECT COUNT(*) FROM trips 
                WHERE arrival_datetime IS NOT NULL 
                AND arrival_datetime < departure_datetime
            """,
            "Consistencia de peso - Trips excediendo capacidad": """
                SELECT COUNT(*) FROM trips t 
                JOIN vehicles v ON t.vehicle_id = v.vehicle_id 
                WHERE t.total_weight_kg > v.capacity_kg
            """,
            "Entregas sin tracking number": """
                SELECT COUNT(*) FROM deliveries 
                WHERE tracking_number IS NULL OR tracking_number = ''
            """
        }
        
        all_valid = True
        for description, query in validations.items():
            self.cursor.execute(query)
            count = self.cursor.fetchone()[0]
            if count > 0:
                logging.warning(f"    {description}: {count} registros")
                all_valid = False
            else:
                logging.info(f"   {description}: OK")
        
        return all_valid
    
    def generate_summary_report(self):
        """Generar reporte resumen de datos generados"""
        logging.info("\n RESUMEN DE GENERACIÓN DE DATOS")
        logging.info("="*50)
        
        tables = ['vehicles', 'drivers', 'routes', 'trips', 'deliveries', 'maintenance']
        total_records = 0
        
        for table in tables:
            self.cursor.execute(f"SELECT COUNT(*) FROM {table}")
            count = self.cursor.fetchone()[0]
            logging.info(f"  {table}: {count:,} registros")
            total_records += count
        
        logging.info(f"\n  TOTAL: {total_records:,} registros")
        
        self.cursor.execute("""
            SELECT 
                AVG(delivery_count) as avg_deliveries_per_trip,
                MIN(delivery_count) as min_deliveries,
                MAX(delivery_count) as max_deliveries
            FROM (
                SELECT trip_id, COUNT(*) as delivery_count
                FROM deliveries
                GROUP BY trip_id
            ) as delivery_stats
        """)
        
        avg_del, min_del, max_del = self.cursor.fetchone()
        logging.info(f"\n  Entregas por viaje: AVG={avg_del:.1f}, MIN={min_del}, MAX={max_del}")
        
        summary = {
            'generation_date': datetime.now().isoformat(),
            'total_records': total_records,
            'table_counts': self.counters,
            'validations_passed': self.validate_data_quality()
        }
        
        with open('generation_summary.json', 'w') as f:
            json.dump(summary, f, indent=2)
        
        logging.info("\n Resumen guardado en generation_summary.json")
    
    def close(self):
        """Cerrar conexión"""
        if self.cursor:
            self.cursor.close()
        if self.connection:
            self.connection.close()
        logging.info("\n Conexión cerrada")


def main():
    """Función principal"""
    print(" FLEETLOGIX - Generación de Datos Masivos")
    print("="*60)
    print("Objetivo: Generar 505000+ registros manteniendo integridad")
    print("="*60)
    
    generator = DataGenerator(DB_CONFIG)
    
    try:
        if not generator.connect():
            return
        
        generator.generate_vehicles(200)
        generator.generate_drivers(400)
        generator.generate_routes(50)
        generator.generate_trips(100000)
        generator.generate_deliveries(400000)
        generator.generate_maintenance(5000)
        
        generator.generate_summary_report()
        
    except Exception as e:
        logging.error(f" Error durante la generación: {e}")
        generator.connection.rollback()
    finally:
        generator.close()


if __name__ == "__main__":
    main()

2026-03-13 22:29:14,170 - INFO -  Conexión exitosa a PostgreSQL
2026-03-13 22:29:14,171 - INFO - Generando 200 vehículos...
2026-03-13 22:29:14,186 - INFO -  200 vehículos insertados
2026-03-13 22:29:14,187 - INFO - Generando 400 conductores...
2026-03-13 22:29:14,236 - INFO -  400 conductores insertados
2026-03-13 22:29:14,237 - INFO - Generando 50 rutas...
2026-03-13 22:29:14,242 - INFO -  50 rutas insertadas
2026-03-13 22:29:14,243 - INFO - Generando 100000 viajes...


 FLEETLOGIX - Generación de Datos Masivos
Objetivo: Generar 505000+ registros manteniendo integridad


Generando trips: 100%|██████████| 100000/100000 [00:02<00:00, 42497.28it/s]
2026-03-13 22:29:16,651 - INFO -   Progreso: 0/100000 trips insertados
2026-03-13 22:29:17,084 - INFO -   Progreso: 10000/100000 trips insertados
2026-03-13 22:29:18,677 - INFO -   Progreso: 20000/100000 trips insertados
2026-03-13 22:29:20,533 - INFO -   Progreso: 30000/100000 trips insertados
2026-03-13 22:29:22,339 - INFO -   Progreso: 40000/100000 trips insertados
2026-03-13 22:29:24,161 - INFO -   Progreso: 50000/100000 trips insertados
2026-03-13 22:29:25,924 - INFO -   Progreso: 60000/100000 trips insertados
2026-03-13 22:29:27,718 - INFO -   Progreso: 70000/100000 trips insertados
2026-03-13 22:29:29,570 - INFO -   Progreso: 80000/100000 trips insertados
2026-03-13 22:29:31,416 - INFO -   Progreso: 90000/100000 trips insertados
2026-03-13 22:29:32,969 - INFO -  100000 viajes insertados
2026-03-13 22:29:32,978 - INFO - Generando 400000 entregas...
Generando deliveries: 100%|█████████▉| 99949/100000 [00:4